[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C06_Interpretability_Course/00_setup/00_environment_check.ipynb)

# 00 · 课程总览与环境自检

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 纯 numpy / matplotlib / pandas + 标准库，无任何其他依赖，所有 cell 秒级跑完。

**本 notebook 你将完成：**

1. **环境自检**：用优雅回退的 `check_import` 检查 numpy / matplotlib / pandas 版本（缺失只标红、不报错）；
2. **热身 A · 最小 residual stream**：用 numpy 构造 embedding + 两个写入分量相加，亲手验证"激活是向量、写入是加法、读取是内积"；
3. **热身 B · 2D 玩具特征空间**：画出两个概念方向与一个激活点的投影，体会"概念是方向"；
4. 完成 3 道 ✏️ 练习：带版本下限的优雅导入检查、向量投影与残差分解、余弦相似度矩阵。

## 1 · 环境自检：`check_import` 的优雅回退写法

本课全程只需要 numpy / matplotlib / pandas。下面的 `check_import` 是评测工程里的标准模式：**依赖检查永远不应该让程序崩溃**——缺什么、缺了要不要紧，打印清楚就好。这个模式在后面模块（以及你自己的 eval harness）里会反复出现。

In [ ]:
import sys
import importlib

def check_import(name, required=True):
    """尝试导入模块并报告版本；失败时优雅回退（打印提示，绝不抛异常）。"""
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, "__version__", "unknown")
        print(f"  ✅ {name:<18s} {ver}")
        return mod
    except ImportError:
        if required:
            print(f"  ❌ {name:<18s} 未安装 —— 本课必需，请 pip install {name}")
        else:
            print(f"  ⚪ {name:<18s} 未安装 —— 本课刻意不依赖它，缺了完全正常")
        return None

print(f"Python {sys.version.split()[0]}  ({sys.platform})\n")

print("本课必需：")
for pkg in ["numpy", "matplotlib", "pandas"]:
    check_import(pkg, required=True)

print("\n本课刻意不依赖（装没装都行，全课纯 numpy 从零实现）：")
for pkg in ["torch", "scipy", "sklearn", "transformer_lens"]:
    check_import(pkg, required=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

rng = np.random.default_rng(0)   # 全课随机数纪律：一律 np.random.default_rng(0)，所有实验可复现

# numpy 冒烟测试：向量化 + 线性代数
x = rng.normal(size=(4, 8))
assert x.shape == (4, 8) and np.isfinite(x).all()

# pandas 冒烟测试
df = pd.DataFrame({"layer": [0, 1, 2], "norm": np.linalg.norm(rng.normal(size=(3, 8)), axis=1)})
print(df.to_string(index=False))

# matplotlib 冒烟测试：能出图即可
fig, ax = plt.subplots(figsize=(4, 2.2))
ax.bar(df["layer"], df["norm"], color="#1f6feb")
ax.set_xlabel("layer"); ax.set_ylabel("||x||"); ax.set_title("matplotlib smoke test")
plt.tight_layout(); plt.show()

print("\n✅ numpy / pandas / matplotlib 全部就绪")

## 2 · 热身 A：最小 residual stream —— 激活是向量，写入是加法

讲解页第 2 节的核心等式：transformer 的最终激活是所有组件写入的**线性叠加**

$$x_{\text{final}} = \underbrace{x_0}_{\text{embedding}} + \sum_\ell \mathrm{Attn}_\ell + \sum_\ell \mathrm{MLP}_\ell$$

下面用 numpy 搭一个最小版本：1 个 embedding + 2 个组件写入（用随机线性映射模拟 attention 和 MLP——真实组件复杂得多，但**接口一模一样**：读取当前 stream，写回一个向量）。重点不是组件内部，而是体会：residual stream 里流动的就是一个普通的 numpy 向量，每个组件的"贡献"可以被单独拿出来核算。

In [ ]:
d_model = 16

# ① embedding：token 进入 residual stream 的初始向量
embed = rng.normal(size=d_model) / np.sqrt(d_model)

# ② 两个组件各自"读取当前 stream，写回一个向量"
W_attn = rng.normal(size=(d_model, d_model)) / np.sqrt(d_model)
W_mlp  = rng.normal(size=(d_model, d_model)) / np.sqrt(d_model)

attn_write = W_attn @ embed                  # attention 读到的是初始 stream
mlp_write  = W_mlp @ (embed + attn_write)    # MLP 读到的是已被 attention 更新过的 stream

# ③ residual stream 的全部秘密：最终激活 = 各分量之和
x_final = embed + attn_write + mlp_write

print("各分量范数：")
for name, v in [("embed", embed), ("attn_write", attn_write),
                ("mlp_write", mlp_write), ("x_final", x_final)]:
    print(f"  {name:<11s} ||v|| = {np.linalg.norm(v):.3f}")

assert np.allclose(x_final, embed + attn_write + mlp_write)   # 严格成立，不是近似
print("\n✅ x_final == embed + attn_write + mlp_write  （激活是向量，写入是加法）")

In [ ]:
# "读取是内积"：取一个 readout 方向（想象成 unembedding 矩阵的一行 = 某个 token 的 logit 方向）
w_readout = rng.normal(size=d_model) / np.sqrt(d_model)

total_logit = x_final @ w_readout

# 内积对加法分配 => 每个分量对 logit 的贡献可以单独核算
contrib = pd.DataFrame({
    "component":    ["embed", "attn_write", "mlp_write"],
    "contribution": [embed @ w_readout, attn_write @ w_readout, mlp_write @ w_readout],
})
contrib["share_of_logit"] = contrib["contribution"] / total_logit
print(contrib.to_string(index=False, float_format=lambda v: f"{v:+.4f}"))

print(f"\n分量贡献之和 = {contrib['contribution'].sum():+.4f}")
print(f"整体 logit    = {total_logit:+.4f}")
assert np.isclose(contrib["contribution"].sum(), total_logit)
print("\n✅ 贡献严格可加 —— 这就是模块 02 要展开的 direct logit attribution 的数学基础")

## 3 · 热身 B：2D 玩具特征空间 —— 概念是方向，读取是投影

线性表示假设（linear representation hypothesis）：概念对应激活空间中的**方向**，概念的强度对应激活在该方向上的**投影长度**。下面在 2D 里画出最小示意：两个概念方向（刻意**不正交**——真实模型里特征方向互相挤压，这是模块 05 叠加 superposition 的主题），一个激活点 = 两个特征的线性叠加 + 噪声。

读出某个特征 = 把激活向概念方向投影。注意观察：因为两个方向不正交，投影读数 ≠ 构造时的真实系数——**特征互相干扰**，这一格的直觉到模块 05 会变成定量分析。

In [ ]:
# 两个概念方向（单位向量，夹角 55 度，刻意不正交）
theta_A, theta_B = np.deg2rad(15), np.deg2rad(70)
d_A = np.array([np.cos(theta_A), np.sin(theta_A)])   # 概念 A：比如"正式语气"
d_B = np.array([np.cos(theta_B), np.sin(theta_B)])   # 概念 B：比如"法语"

# 一个激活点：特征 A 强度 1.8 + 特征 B 强度 1.1 + 一点噪声
coef_A, coef_B = 1.8, 1.1
x_act = coef_A * d_A + coef_B * d_B + rng.normal(scale=0.05, size=2)

def proj_len(v, d):
    """v 在单位方向 d 上的投影长度（读出特征强度）"""
    return float(v @ d)

fig, ax = plt.subplots(figsize=(5.5, 5))
for d, name, c in [(d_A, "concept A", "#1f6feb"), (d_B, "concept B", "#d29922")]:
    ax.annotate("", xy=tuple(3.0 * d), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=c, lw=2))
    ax.text(*(3.1 * d), name, color=c, fontsize=11)
    L = proj_len(x_act, d)
    p = L * d                                         # 投影点
    ax.plot([x_act[0], p[0]], [x_act[1], p[1]], ls="--", c=c, alpha=0.7)
    ax.plot(*p, "o", c=c, ms=6)
    ax.text(*(p + np.array([0.07, -0.12])), f"proj = {L:.2f}", color=c, fontsize=9)

ax.plot(*x_act, "k*", ms=15)
ax.text(x_act[0] + 0.08, x_act[1] + 0.08, "activation x", fontsize=11)
ax.set_xlim(-0.3, 3.4); ax.set_ylim(-0.3, 3.4); ax.set_aspect("equal")
ax.grid(alpha=0.3)
ax.set_title("toy feature space: concepts are directions,\nreading a feature = projection")
plt.tight_layout(); plt.show()

print(f"构造时的真实系数      : A = {coef_A:.2f}, B = {coef_B:.2f}")
print(f"投影读出的特征强度    : A = {proj_len(x_act, d_A):.2f}, B = {proj_len(x_act, d_B):.2f}")
print("两者不相等！方向不正交 => 特征互相干扰 —— 模块 05（superposition/SAE）的出发点")

---
## ✏️ 练习 1：实现 `check_import_ex` —— 带版本下限的优雅导入检查

把第 1 节的模式升级成可复用的工具函数。实现 `check_import_ex(name, min_version=None)`，返回二元组 `(ok, version)`：

- 模块不存在 → `(False, None)`，**不允许抛任何异常**；
- 模块存在、`min_version is None` → `(True, 版本字符串)`；
- 模块存在、给了 `min_version` → 版本 ≥ `min_version` 时 `ok=True`，否则 `ok=False`（`version` 仍返回真实版本字符串）；
- 模块存在但没有 `__version__` 属性 → 版本按 `"0"` 处理。

**提示**：`importlib.import_module` + `try/except ImportError`。版本比较**不要直接比字符串**（`"1.10" < "1.9"` 在字符串序下是 True，这是错的）——先写辅助函数 `_version_tuple`：按 `.` 切开，每段取开头的连续数字转 int（如 `"4rc1"` → 4；整段没数字记 0），组成 tuple 再比。两个函数合计 15–20 行。

In [ ]:
import importlib
import re

def _version_tuple(ver):
    # TODO: "1.26.4" -> (1, 26, 4)；"2.3.0rc1" -> (2, 3, 0)；非数字开头的段记 0
    raise NotImplementedError

def check_import_ex(name, min_version=None):
    # TODO: 按练习说明实现；永不抛异常，返回 (ok, version)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
ok, ver = check_import_ex("numpy")
assert ok is True and isinstance(ver, str) and len(ver) > 0
assert check_import_ex("a_module_that_does_not_exist_xyz") == (False, None)   # 缺失：不抛异常
ok, ver = check_import_ex("numpy", min_version="1.0")
assert ok is True
ok, ver = check_import_ex("numpy", min_version="999.0")
assert ok is False and isinstance(ver, str)            # 版本不够：ok=False，但版本照样返回
assert _version_tuple("1.26.4") == (1, 26, 4)
assert _version_tuple("1.10") > _version_tuple("1.9")  # 字符串序会判错，tuple 序必须判对
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `project` —— 投影分量与残差分解

实现 `project(v, d)`：把向量 `v` 分解为"沿方向 `d` 的分量"与"垂直于 `d` 的残差"，返回 `(proj, resid)`：

$$\mathrm{proj} = \frac{v \cdot d}{d \cdot d}\, d, \qquad \mathrm{resid} = v - \mathrm{proj}$$

这是全课的原子操作：模块 01 用它说明探针在读哪个方向，模块 06 的 steering / directional ablation 就是对 `proj` 做手术。

**提示**：3–5 行。注意 `d` **不保证是单位向量**——公式里除以 $d \cdot d$ 正是在处理这一点（顺带获得"对 `d` 缩放不变"的性质）。`proj + resid == v` 与 `proj ⊥ resid` 由公式自动保证，自测会逐条验证。

In [ ]:
def project(v, d):
    # TODO: 返回 (proj, resid)，其中 proj = (v·d / d·d) * d，resid = v - proj
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
rng_t = np.random.default_rng(0)
v = rng_t.normal(size=16)
d = 3.7 * rng_t.normal(size=16)                 # 故意不是单位向量
proj, resid = project(v, d)

assert np.allclose(proj + resid, v)             # 分解完备：两分量加回去 = 原向量
assert abs(float(resid @ d)) < 1e-10            # 残差 ⊥ 方向
assert abs(float(proj @ resid)) < 1e-10         # 两分量正交

p2, r2 = project(v, 2.0 * d)
assert np.allclose(p2, proj) and np.allclose(r2, resid)   # 对 d 缩放不变

p3, r3 = project(d, d)
assert np.allclose(p3, d) and np.allclose(r3, np.zeros_like(d))   # v ∥ d：残差为零
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 `cosine_sim_matrix` —— 余弦相似度矩阵

实现 `cosine_sim_matrix(X)`：输入形状 `(n, d)` 的矩阵（n 个 d 维向量，每行一个），返回 `(n, n)` 矩阵 `S`，`S[i, j]` 为第 i、j 行的余弦相似度

$$S_{ij} = \frac{X_i \cdot X_j}{\lVert X_i \rVert \, \lVert X_j \rVert}$$

模块 03 比较 attention 头、模块 05 比较 SAE 特征方向、模块 06 比较 steering 向量，全靠它。

**提示**：3–5 行，**禁止双重 for 循环**。先按行归一化 `Xn = X / np.linalg.norm(X, axis=1, keepdims=True)`，然后一个矩阵乘法 `Xn @ Xn.T` 就是答案；最后 `np.clip(S, -1.0, 1.0)` 防浮点误差越界（假定输入没有零向量）。

In [ ]:
def cosine_sim_matrix(X):
    # TODO: 行归一化 -> 矩阵乘法 -> clip 到 [-1, 1]，返回 (n, n) 相似度矩阵
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
S = cosine_sim_matrix(np.eye(3))
assert S.shape == (3, 3) and np.allclose(S, np.eye(3))    # 正交单位基：对角 1，其余 0

X = np.array([[1.0, 0.0], [2.0, 0.0], [-3.0, 0.0], [0.0, 5.0]])
S = cosine_sim_matrix(X)
assert np.allclose(np.diag(S), 1.0)                       # 自相似恒为 1，与长度无关
assert np.isclose(S[0, 1], 1.0)                           # 同向：+1
assert np.isclose(S[0, 2], -1.0)                          # 反向：-1
assert np.isclose(S[0, 3], 0.0)                           # 垂直：0
assert np.allclose(S, S.T)                                # 对称

R = np.random.default_rng(0).normal(size=(20, 8))
S = cosine_sim_matrix(R)
assert S.shape == (20, 20) and S.max() <= 1.0 and S.min() >= -1.0   # clip 后严格落在 [-1, 1]
print("✅ 练习 3 通过")

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
import importlib
import re

def _version_tuple(ver):
    parts = []
    for seg in str(ver).split("."):
        m = re.match(r"\d+", seg)
        parts.append(int(m.group()) if m else 0)
    return tuple(parts)

def check_import_ex(name, min_version=None):
    try:
        mod = importlib.import_module(name)
    except ImportError:
        return (False, None)
    ver = getattr(mod, "__version__", "0")
    ok = min_version is None or _version_tuple(ver) >= _version_tuple(min_version)
    return (ok, ver)

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def project(v, d):
    proj = (v @ d) / (d @ d) * d
    return proj, v - proj

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def cosine_sim_matrix(X):
    Xn = X / np.linalg.norm(X, axis=1, keepdims=True)
    return np.clip(Xn @ Xn.T, -1.0, 1.0)

---
## 小结

- **环境**：本课只需 numpy / matplotlib / pandas；`check_import` 的优雅回退是依赖检查的标准姿势——报告问题，不制造问题。
- **激活是向量，写入是加法**：residual stream 的最终激活 = embedding + 各组件写入之严格线性叠加（[Elhage 2021] 的求和分解），所以每个组件对任意 readout 方向的贡献都可单独核算——模块 02 的 direct logit attribution 由此而来。
- **概念是方向，读取是投影**：线性表示假设下，读特征 = 内积/投影；概念方向不正交时投影读数会互相污染——模块 05 superposition/SAE 的全部动机已经藏在那张 2D 图里。
- 你写好的 `project` 与 `cosine_sim_matrix` 会在模块 01/03/05/06 被直接复用，请保管好。

**下一步 → 模块 01 · Probing：用线性探针读激活**——把"读取是内积"升级成可训练的探针：特征在不在、在哪层、有多线性，以及探针自己会作弊的全部方式。